<a href="https://colab.research.google.com/github/JSJeong-me/MCP/blob/main/00%20langchain_agent_tool_calling_intro_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain Agent Tool Calling 입문 실습 (Colab)

이 노트북은 **MCP를 이해하기 위한 사전 단계**로,
먼저 **LangChain agent가 tool을 어떻게 호출하는지**를 가장 작은 예제로 보여줍니다.

## 이 실습에서 배우는 것

- `@tool` 로 **LangChain tool** 을 정의하는 방법
- `create_agent(...)` 로 **agent** 를 만드는 방법
- 사용자의 요청에 따라 agent가 **어떤 tool을 호출할지 결정**하는 흐름
- 왜 이 구조가 나중에 **MCP tool 호출 구조와 거의 같아지는지**

## 핵심 아이디어

지금 실습에서는 tool이 **로컬 Python 함수**입니다.
나중에 MCP를 붙이면 이 tool이 **외부 MCP 서버가 제공하는 tool**로 바뀝니다.

즉, 개념상 흐름은 같습니다.

- 지금: `tools=[파이썬 함수들]`
- MCP 사용 시: `tools=await mcp_client.get_tools()`


## 실행 순서

1. **필수 패키지 설치**
2. **OpenAI API Key 설정**
3. **LangChain tool 정의**
4. **agent 생성**
5. **첫 번째 사용자 요청 실행**
6. **두 번째 사용자 요청 실행**
7. **전체 메시지 흐름 확인**




In [2]:
# 1) 패키지 설치
!pip -q install -U langchain langchain-openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.0/513.0 kB 3.9 MB/s eta 0:00:00


In [3]:
import os
from google.colab import userdata

# Colab 비밀 저장소에서 OPENAI_API_KEY 가져오기
openai_api_key_from_secrets = userdata.get('OPENAI_API_KEY')

if openai_api_key_from_secrets:
    os.environ["OPENAI_API_KEY"] = openai_api_key_from_secrets
    print("OPENAI_API_KEY가 Colab 비밀 저장소에서 설정되었습니다.")
else:
    print("Colab 비밀 저장소에 'OPENAI_API_KEY'가 없습니다. 다음 셀에서 수동으로 입력하거나 환경 변수를 확인해주세요.")

OPENAI_API_KEY가 Colab 비밀 저장소에서 설정되었습니다.


## 1) Tool 정의

LangChain에서 tool은 보통 `@tool` 데코레이터로 정의합니다.

이 예제에서는 두 개의 tool을 만듭니다.

1. `get_weather(city)`
   - 도시를 입력받아 간단한 날씨 설명을 반환

2. `suggest_opening_line(weather_description, genre)`
   - 날씨 설명과 장르를 받아 소설 첫 문장을 생성


In [4]:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """도시 이름을 받아 간단한 날씨 정보를 반환합니다."""
    weather_map = {
        "서울": "서울은 화창하고 벚꽃이 피기 좋은 날씨입니다.",
        "부산": "부산은 바닷바람이 부드럽고 맑은 날씨입니다.",
        "제주": "제주는 옅은 안개와 함께 감성적인 새벽 분위기입니다.",
    }
    return weather_map.get(city, f"{city}의 날씨는 소설 속 배경으로 쓰기 좋은 분위기입니다.")

@tool
def suggest_opening_line(weather_description: str, genre: str) -> str:
    """날씨 설명과 장르를 받아 소설의 첫 문장을 한 줄 생성합니다."""
    if genre == "romance":
        return f"{weather_description} 그녀를 처음 마주한 순간, 계절은 비밀처럼 천천히 피어나고 있었다."
    elif genre == "mystery":
        return f"{weather_description} 그날의 공기에는 설명할 수 없는 불길한 침묵이 스며 있었다."
    else:
        return f"{weather_description} 그 풍경은 새로운 이야기가 시작되기에 충분히 아름다웠다."

print("Tool 정의 완료")


Tool 정의 완료


## 2) Agent 생성

이제 최신 LangChain 방식인 `create_agent(...)` 로 agent를 생성합니다.

이 agent는:
- 사용자의 요청을 읽고
- 필요하면 `get_weather` tool을 호출하고
- 필요하면 `suggest_opening_line` tool을 호출한 뒤
- 최종 답변을 한국어로 작성합니다.


In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-4.1",
    tools=[get_weather, suggest_opening_line],
    system_prompt=(
        "당신은 초보 작가를 돕는 창의적인 소설 도우미입니다. "
        "날씨 정보가 필요하면 get_weather tool을 사용하세요. "
        "첫 문장을 만들 때는 suggest_opening_line tool을 사용하세요. "
        "최종 답변은 한국어로 자연스럽게 작성하세요."
    ),
)

print("Agent 생성 완료")


Agent 생성 완료


## 3) 첫 번째 요청 실행

질문:
> 지금 서울 날씨를 참고해서, 로맨스 소설의 첫 문장을 한 줄 써줘.


In [6]:
user_input = "지금 서울 날씨를 참고해서, 로맨스 소설의 첫 문장을 한 줄 써줘."

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": user_input}
        ]
    }
)

print("=== 전체 메시지 흐름 ===")
for i, msg in enumerate(result["messages"]):
    print(f"\n[{i}] type={getattr(msg, 'type', type(msg).__name__)}")
    print("content:", getattr(msg, "content", ""))

print("\n=== 최종 답변 ===")
print(result["messages"][-1].content)


=== 전체 메시지 흐름 ===

[0] type=human
content: 지금 서울 날씨를 참고해서, 로맨스 소설의 첫 문장을 한 줄 써줘.

[1] type=ai
content: 

[2] type=tool
content: 서울은 화창하고 벚꽃이 피기 좋은 날씨입니다.

[3] type=ai
content: 

[4] type=tool
content: 화창하고 벚꽃이 피기 좋은 날씨 그 풍경은 새로운 이야기가 시작되기에 충분히 아름다웠다.

[5] type=ai
content: 화창하고 벚꽃이 피기 좋은 날씨, 그 풍경은 새로운 이야기가 시작되기에 충분히 아름다웠다.

=== 최종 답변 ===
화창하고 벚꽃이 피기 좋은 날씨, 그 풍경은 새로운 이야기가 시작되기에 충분히 아름다웠다.


## 4) 두 번째 요청 실행

질문:
> 제주 날씨를 참고해서 미스터리 소설 첫 문장을 한 줄 써줘.


In [7]:
user_input = "제주 날씨를 참고해서 미스터리 소설 첫 문장을 한 줄 써줘."

result2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": user_input}
        ]
    }
)

print("=== 최종 답변 ===")
print(result2["messages"][-1].content)


=== 최종 답변 ===
옅은 안개가 깔린 감성적인 새벽, 누구도 예상치 못한 사건이 그 조용한 거리에 스며들었다.


## 5) 왜 이게 MCP 이해에 중요한가?

지금은 tool이 **로컬 Python 함수**입니다.

```python
tools = [get_weather, suggest_opening_line]
```

나중에 MCP를 사용하면 보통 이렇게 바뀝니다.

```python
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({...})
tools = await client.get_tools()
```

즉, agent 입장에서는 여전히 **“tool 목록을 받아서 호출한다”**는 점이 같습니다.

차이는 단 하나입니다.

- 지금: tool이 노트북 안의 함수
- MCP 사용 시: tool이 외부 MCP 서버가 제공하는 함수

그래서 이 노트북은 **MCP를 이해하기 위한 가장 좋은 사전 실습**입니다.


## 6) 실습 정리

이번 실습에서 확인한 핵심:

1. Tool은 agent가 호출할 수 있는 함수입니다.
2. Agent는 사용자 요청을 보고 어떤 tool을 쓸지 결정합니다.
3. Tool 호출 결과를 바탕으로 최종 답변을 생성합니다.
4. MCP는 이런 tool을 **외부 서버 표준 방식**으로 가져오게 해 주는 계층입니다.

다음 단계로 확장하려면:
- 이 로컬 tool 예제를 **FastMCP 서버**로 바꾸고
- LangChain의 **MCP client** 로 불러오면 됩니다.
